apapla split karun ghya cells

In [ ]:
import numpy as np

# =====================================================
# 1. ENVIRONMENT SETTINGS
# =====================================================

grid_size = 5

goal_position = (4, 4)

actions = [
    "up",
    "down",
    "left",
    "right"
]

num_actions = len(actions)

# =====================================================
# 2. Q-LEARNING PARAMETERS
# =====================================================

alpha = 0.1      # learning rate
gamma = 0.9      # discount factor
epsilon = 0.2    # exploration rate

episodes = 100

# =====================================================
# 3. CREATE AGENTS
# =====================================================

num_agents = 3

# Each agent has its own Q-table
agent_qtables = []

for i in range(num_agents):

    q_table = np.zeros((grid_size, grid_size, num_actions))

    agent_qtables.append(q_table)

# =====================================================
# 4. ACTION FUNCTION
# =====================================================

def take_action(state, action):

    x, y = state

    if action == 0 and x > 0:
        x -= 1

    elif action == 1 and x < grid_size - 1:
        x += 1

    elif action == 2 and y > 0:
        y -= 1

    elif action == 3 and y < grid_size - 1:
        y += 1

    return (x, y)

# =====================================================
# 5. REWARD FUNCTION
# =====================================================

def get_reward(state):

    if state == goal_position:
        return 10

    return -1

# =====================================================
# 6. LOCAL TRAINING
# =====================================================

for agent in range(num_agents):

    print(f"\nTraining Agent {agent+1}")

    q_table = agent_qtables[agent]

    for episode in range(episodes):

        state = (0, 0)

        done = False

        while not done:

            x, y = state

            # Epsilon-greedy action selection
            if np.random.rand() < epsilon:

                action = np.random.randint(num_actions)

            else:

                action = np.argmax(q_table[x, y])

            # Take action
            next_state = take_action(state, action)

            reward = get_reward(next_state)

            nx, ny = next_state

            # =================================================
            # Q-LEARNING UPDATE
            # =================================================

            old_value = q_table[x, y, action]

            next_max = np.max(q_table[nx, ny])

            new_value = old_value + alpha * (
                reward + gamma * next_max - old_value
            )

            q_table[x, y, action] = new_value

            # Move to next state
            state = next_state

            # Goal reached
            if state == goal_position:
                done = True

    # Save trained table
    agent_qtables[agent] = q_table

# =====================================================
# 7. FEDERATED AVERAGING
# =====================================================

print("\nPerforming Federated Averaging")

global_qtable = np.mean(agent_qtables, axis=0)

# =====================================================
# 8. DISTRIBUTE GLOBAL MODEL
# =====================================================

for i in range(num_agents):

    agent_qtables[i] = global_qtable.copy()

print("Global Q-table distributed to all agents")

# =====================================================
# 9. TEST GLOBAL AGENT
# =====================================================

print("\nTesting Global Agent")

state = (0, 0)

steps = 0

while state != goal_position and steps < 20:

    x, y = state

    action = np.argmax(global_qtable[x, y])

    next_state = take_action(state, action)

    print(f"State: {state} -> Action: {actions[action]}")

    state = next_state

    steps += 1

print("\nFinal State:", state)

if state == goal_position:
    print("Goal Reached Successfully")
else:
    print("Goal Not Reached")